# InstruDetector – Complete ML Pipeline

**Training Results & Evaluation Notebook**

This notebook includes:
- Complete training results (27 epochs)
- Full evaluation pipeline
- Predictions, confusion matrix, feature maps
- Model saving/loading capabilities

**Classes:** Guitar, Piano, Mallet, String instruments

## Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import random
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## Model Architecture

In [ ]:
class SimpleAudioCNN(nn.Module):
    def __init__(self, n_mels=128, n_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.3)
        # Fully connected layers are initialized after the first forward pass
        self.fc1 = None
        self.fc2 = None
        self.n_classes = n_classes

    def forward(self, x):
        # Input: (batch, 1, n_mels, time)
        assert x.ndim == 4, f"Expected 4D input (batch, 1, n_mels, time), got {x.shape}"
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        # Initialize fully connected layers on first pass
        if self.fc1 is None:
            self.fc1 = nn.Linear(x.shape[1], 128).to(x.device)
            self.fc2 = nn.Linear(128, self.n_classes).to(x.device)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize model
model = SimpleAudioCNN(n_mels=128, n_classes=4).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model Architecture Summary:')
print(f'  • Total Parameters: {total_params:,}')
print(f'  • Input: Mel spectrograms (1 channel)')
print(f'  • Output: 4 classes (Guitar, Piano, Mallet, String)')
print(f'  • Architecture: 3 Conv blocks + Adaptive Pooling + 2 FC layers')

## Training Results Summary

In [ ]:
# Training history from run_20250609-150425
train_losses = [0.7181073010395721, 0.5083118911561673, 0.4290516093656276, 0.3851754194787251, 0.3563204851689964, 0.3338076468572649, 0.31717919112338877, 0.2981249919491628, 0.2884761876460715, 0.27818278772320776, 0.2697861295464333, 0.25927542196608294, 0.2536339473053502, 0.2481031457636024, 0.24041982632408185, 0.2360725552749515, 0.2309605563210428, 0.2276193623471144, 0.22217582252813523, 0.21900674211307358, 0.2159190010080988, 0.21007855815339305, 0.20843149196734412, 0.20526236967080552, 0.2028134335483795, 0.19890849243274936, 0.19711411663086506]
train_accs = [71.71, 80.89, 84.14, 85.71, 86.94, 87.72, 88.28, 89.06, 89.40, 89.88, 90.07, 90.53, 90.77, 91.02, 91.29, 91.47, 91.62, 91.73, 92.01, 92.12, 92.13, 92.35, 92.47, 92.52, 92.64, 92.81, 92.89]
valid_losses = [0.6323581601060985, 0.5779308084672188, 0.5833318001449768, 0.5419294424042611, 0.6727474016736954, 0.5399571747557503, 0.8677940129913847, 0.6575690654451037, 0.5377633120740911, 0.6799338925848548, 0.47443475178606914, 0.5590065504552053, 0.5744452592671537, 0.5749251242253414, 0.6070893277501308, 0.8452185352789959, 0.5103502303051972, 0.6709305661430858, 0.6501142074701091, 0.6827143148382572, 0.9398382467057472, 0.7521884607932824, 0.5553852463280423, 0.5328214466271886, 0.7262707753534966, 0.7359758130818636, 0.5866634988096967]
valid_accs = [75.24, 79.17, 76.12, 79.91, 73.65, 76.38, 71.89, 76.70, 77.98, 75.60, 81.40, 79.40, 79.15, 79.87, 78.63, 73.67, 81.80, 78.73, 78.72, 77.94, 73.05, 77.44, 81.08, 81.53, 77.62, 78.28, 80.71]

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
epochs_range = range(1, len(train_losses) + 1)
ax1.plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, valid_losses, 'r-', label='Valid Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(epochs_range, train_accs, 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, valid_accs, 'r-', label='Valid Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final Training Results (Epoch 27):')
print(f'  • Training Loss: {train_losses[-1]:.4f}')
print(f'  • Training Accuracy: {train_accs[-1]:.2f}%')
print(f'  • Validation Loss: {valid_losses[-1]:.4f}')
print(f'  • Validation Accuracy: {valid_accs[-1]:.2f}%')
print(f'\nBest Validation Accuracy: {max(valid_accs):.2f}% (Epoch {valid_accs.index(max(valid_accs)) + 1})')

## Confusion Matrix & Classification Report

In [ ]:
# Load the model and generate predictions
model = SimpleAudioCNN(n_mels=128, n_classes=4).to(device)

# Initialize FC layers with a dummy forward pass
# Input shape: (batch, 1, n_mels, time) where n_mels=128
dummy_input = torch.randn(1, 1, 128, 120).to(device)  # Using 120 time steps to match training data
with torch.no_grad():
    # Print shapes after each layer
    x = dummy_input
    print(f'Input shape: {x.shape}')
    x = model.pool(F.relu(model.bn1(model.conv1(x))))
    print(f'After conv1: {x.shape}')
    x = model.pool(F.relu(model.bn2(model.conv2(x))))
    print(f'After conv2: {x.shape}')
    x = model.pool(F.relu(model.bn3(model.conv3(x))))
    print(f'After conv3: {x.shape}')
    x = model.dropout(x)
    x = torch.flatten(x, 1)
    print(f'After flatten: {x.shape}')
    _ = model(dummy_input)  # This will initialize fc1 and fc2

# Now load the weights
model_path = '../outputs/run_20250609-150425/checkpoints/model_e17_acc81.8_20250609-194111.pt'
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

# Create validation dataset and loader
val_mel_dir = '../data/mel_spectrograms/valid'
val_json = '../data/json/nsynth-valid-filtered.json'
val_dataset = MelSpecDataset(val_mel_dir, val_json)
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Generate predictions on validation set
all_preds = []
all_labels = []

with torch.no_grad():
    for data, target in valid_loader:
        data = data.to(device)
        output = model(data)
        preds = output.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(target.numpy())

# Class names
class_names = ['guitar', 'piano', 'mallet', 'string']

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate accuracy
total_accuracy = 100 * np.sum(np.array(all_labels) == np.array(all_preds)) / len(all_labels)
print(f'Overall Accuracy: {total_accuracy:.1f}%\n')

# Classification Report
print('DETAILED CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(all_labels, all_preds, 
                          target_names=class_names, digits=3))

## Results Summary & Conclusions

### Model Performance:
- **Final Validation Accuracy**: 80.71%
- **Best Validation Accuracy**: 81.80% (Epoch 17)
- **Training Accuracy**: 92.89%
- **Training Loss**: 0.1971
- **Validation Loss**: 0.5867

### Key Observations:
1. **Training Progress**:
   - Model shows good learning progression
   - Training accuracy steadily increased to 92.89%
   - Best validation accuracy achieved at epoch 17

2. **Overfitting Signs**:
   - Gap between training (92.89%) and validation (80.71%) accuracy
   - Validation accuracy fluctuates more than training accuracy

3. **Model Architecture**:
   - 3-layer CNN with batch normalization
   - Adaptive pooling for variable input sizes
   - Dropout (0.5) for regularization

### Recommendations for Improvement:
1. **Regularization**:
   - Increase dropout rate
   - Add L2 regularization
   - Implement data augmentation

2. **Architecture**:
   - Try deeper networks (ResNet, EfficientNet)
   - Experiment with attention mechanisms
   - Consider ensemble methods

3. **Training**:
   - Implement learning rate scheduling
   - Use early stopping
   - Try different optimizers

### Next Steps:
1. Implement suggested improvements
2. Collect more diverse training data
3. Experiment with different architectures
4. Add real-time inference capabilities